## Loading the dataset

In [1]:
import pandas as pd
import numpy as np

In [2]:

path = "dataset/Original/global/global-genre_network-2019.csv"
df = pd.read_csv(path, sep="\t")

print(df.head)


<bound method NDFrame.head of               source          target  weight   avg_streams
0              latin       reggaeton     479  1.060468e+08
1              latin           latin     282  1.048953e+08
2            pop rap             rap     222  5.385630e+07
3                rap            trap     218  6.656220e+07
4            hip hop             rap     203  5.405006e+07
...              ...             ...     ...           ...
2923   latin hip hop      latin rock       1  4.541218e+07
2924      latin rock  reggaeton flow       1  4.541218e+07
2925   latin hip hop     salsa choke       1  4.541218e+07
2926  reggaeton flow     salsa choke       1  4.541218e+07
2927         bachata  electro latino       1  1.053271e+08

[2928 rows x 4 columns]>


## Building a co-occurance matrix 

In [3]:
genres = pd.unique(df[["source", "target"]].values.ravel())

# mapping: genre → index
genre_to_idx = {g: i for i, g in enumerate(genres)}

n = len(genres)
adj_matrix = np.zeros((n, n), dtype=float)

for _, row in df.iterrows():
    i = genre_to_idx[row["source"]]
    j = genre_to_idx[row["target"]]
    adj_matrix[i, j] = row["weight"]  

### matrix check

In [4]:
print("Matrix shape:", adj_matrix.shape)
max_value = adj_matrix.max()
print("Max weight:", max_value)

Matrix shape: (310, 310)
Max weight: 479.0


### creating embeddings

In [ ]:
# making the matrix sparse (?)
from scipy.sparse import csr_matrix
genre_graph = csr_matrix(adj_matrix)

In [6]:
datasets = {
    "genre_graph": genre_graph,
}

In [7]:
# ################### Binary Tree
depth = 10 
N = sum(2 ** i for i in range(depth))
mat = np.zeros((N, N))
for i in range(N):
    j = 2 * i + 1
    if j + 1 >= N:
        break
    mat[i, j] = 1
    mat[i, j + 1] = 1
binary_tree = csr_matrix(mat)

# ################### Quad Tree
depth = 4
N = sum(4 ** i for i in range(depth))
mat = np.zeros((N, N))
for i in range(N):
    j = 4 * i + 1
    if j + 3 >= N:
        break
    mat[i, j] = 1
    mat[i, j + 1] = 1
    mat[i, j + 2] = 1
    mat[i, j + 3] = 1
quad_tree = csr_matrix(mat)

datasets["binary_tree"] = binary_tree
datasets["quad_tree"] = quad_tree

In [8]:
def get_dataset(dataset_name_or_filename):
    print("using {} dataset".format(dataset_name_or_filename))
    if dataset_name_or_filename in datasets:
        print("using builtin dataset")
        return datasets[dataset_name_or_filename]
    else:
        return pickle.load(open(dataset_name_or_filename, 'rb'))

In [9]:
get_dataset("genre_graph")

using genre_graph dataset
using builtin dataset


<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 2928 stored elements and shape (310, 310)>